# New Zealand Treasury Fiscal Strategy Model (FSM) Baseline Replicator & Scenario Engine

This project builds a quantitative simulation engine to replicate the New Zealand Treasury's Fiscal Strategy Model (FSM). It models the 10-year trajectory of the Operating Balance Before Gains and Losses (OBEGAL) and Net Core Crown Debt. By incorporating dynamic revenue elasticities, compounding budget operating allowances, debt servicing transmission, and stochastic GDP growth shocks, this tool evaluates fiscal sustainability and the direct impact of government spending choices on the Crown's return to surplus.

---

### Core Mathematical Framework

#### 1. Nominal GDP Dynamics
Nominal GDP ($Y_t$) drives tax revenue generation and serves as the baseline denominator for sovereign debt ratios:
$$Y_t = Y_{t-1} \cdot (1 + g_t^{nom})$$
where $g_t^{nom} = (1 + g_t^{real})(1 + \pi_t) - 1$, $g_t^{real}$ is real GDP growth, and $\pi_t$ is the GDP deflator inflation rate.

#### 2. Core Crown Tax Revenue Model
Core Crown tax revenue ($R_t$) is modeled via revenue buoyancy ($\varepsilon_{rev}$), capturing tax elasticity relative to nominal output:
$$R_t = R_{t-1} \cdot \left[ 1 + \varepsilon_{rev} \cdot g_t^{nom} \right]$$

#### 3. Core Crown Expenditure Framework
Total Core Crown expenses ($E_t$) comprise baseline operating expenses ($E_t^{base}$), cumulative new budget operating allowances ($A_t^{cum}$), and debt servicing costs ($S_t$):
$$E_t^{base} = E_{t-1}^{base} \cdot (1 + \pi_t) + \text{Demographic Indexation}_t$$
$$A_t^{cum} = A_{t-1}^{cum} + \text{Allowance}_t$$
$$S_t = D_{t-1} \cdot i_t^{effective}$$
$$E_t^{total} = E_t^{base} + A_t^{cum} + S_t$$

#### 4. OBEGAL (Operating Balance Before Gains and Losses)
$$OBEGAL_t = R_t + R_t^{non-tax} - E_t^{total}$$

#### 5. Net Core Crown Debt Accumulation
Net Core Crown Debt ($D_t$) evolves based on the negative OBEGAL deficit, net capital spending allocations ($K_t$), and working capital liquidity adjustments ($\Delta W_t$):
$$D_t = D_{t-1} - OBEGAL_t + K_t + \Delta W_t$$
$$\text{Net Debt Ratio}_t = \left( \frac{D_t}{Y_t} \right) \times 100$$

In [1]:
# Cell 2: Installs and Imports

import sys
import subprocess

# Ensure required packages are installed
required_packages = ['pandas', 'numpy', 'plotly', 'scipy', 'statsmodels']
for package in required_packages:
    try:
        __import__(package)
    except ImportError:
        subprocess.check_call([sys.executable, "-m", "pip", "install", package])

import numpy as np
import pandas as pd
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import plotly.express as px
from scipy.stats import norm

print("All fiscal modeling dependencies successfully loaded.")

All fiscal modeling dependencies successfully loaded.


In [2]:
# Cell 3: Data Pipeline - Baseline Fiscal & Macroeconomic Parameters

def load_nz_treasury_baseline_data():
    """
    Constructs the baseline macro-fiscal starting parameters reflecting
    recent NZ Treasury BEFU (Budget Economic and Fiscal Update) datasets.
    """
    # Base Year Parameters (FY2025/2026 Starting Point)
    base_year = 2026
    projection_years = 10
    years = [base_year + i for i in range(projection_years + 1)]

    # Starting Fiscal Aggregates (in NZD Millions)
    base_gdp = 425000.0         # Nominal GDP (~$425 Billion NZD)
    base_tax_revenue = 118000.0  # Core Crown Tax Revenue
    base_nontax_revenue = 12500.0# Non-tax revenue (Crown Entities, ACC, Fees)
    base_expenses = 138000.0    # Core Crown Baseline Expenses (excl. debt servicing & new allowances)
    base_net_debt = 175000.0    # Net Core Crown Debt (~41.2% of GDP)

    # Macroeconomic Assumptions (10-Year Projections)
    # Baseline growth paths: Transition from lower current growth to trend ~2.3% real, ~2.0% inflation
    real_gdp_growth = np.array([0.012, 0.021, 0.024, 0.025, 0.023, 0.023, 0.023, 0.022, 0.022, 0.022, 0.022])
    inflation_rate  = np.array([0.027, 0.022, 0.020, 0.020, 0.020, 0.020, 0.020, 0.020, 0.020, 0.020, 0.020])
    effective_interest_rate = np.array([0.042, 0.043, 0.044, 0.043, 0.042, 0.041, 0.040, 0.040, 0.040, 0.040, 0.040])

    # Elasticities and Multipliers
    tax_buoyancy = 1.05          # Elasticity of tax revenue relative to nominal GDP growth
    capital_allowance_annual = 4500.0 # Net Annual Capital Allocations ($4.5B/yr)

    baseline_df = pd.DataFrame({
        'Fiscal_Year': years,
        'Real_GDP_Growth': real_gdp_growth,
        'Inflation': inflation_rate,
        'Nominal_GDP_Growth': (1 + real_gdp_growth) * (1 + inflation_rate) - 1,
        'Effective_Interest_Rate': effective_interest_rate
    })

    base_dict = {
        'base_year': base_year,
        'base_gdp': base_gdp,
        'base_tax_revenue': base_tax_revenue,
        'base_nontax_revenue': base_nontax_revenue,
        'base_expenses': base_expenses,
        'base_net_debt': base_net_debt,
        'tax_buoyancy': tax_buoyancy,
        'capital_allowance_annual': capital_allowance_annual,
        'macro_df': baseline_df
    }

    return base_dict

# Initialize and display baseline parameters
baseline_data = load_nz_treasury_baseline_data()
print("Baseline Macro-Fiscal Parameters Successfully Ingested:")
print(baseline_data['macro_df'][['Fiscal_Year', 'Real_GDP_Growth', 'Inflation', 'Nominal_GDP_Growth']].to_string(index=False))

Baseline Macro-Fiscal Parameters Successfully Ingested:
 Fiscal_Year  Real_GDP_Growth  Inflation  Nominal_GDP_Growth
        2026            0.012      0.027            0.039324
        2027            0.021      0.022            0.043462
        2028            0.024      0.020            0.044480
        2029            0.025      0.020            0.045500
        2030            0.023      0.020            0.043460
        2031            0.023      0.020            0.043460
        2032            0.023      0.020            0.043460
        2033            0.022      0.020            0.042440
        2034            0.022      0.020            0.042440
        2035            0.022      0.020            0.042440
        2036            0.022      0.020            0.042440


In [3]:
# Cell 4: Fiscal Strategy Model (FSM) Engine Class

class NZFiscalStrategyModel:
    """
    Quantitative simulation engine replicating the NZ Treasury Fiscal Strategy Model.
    Models OBEGAL, Net Core Crown Debt, tax buoyancies, and budget spending allowances.
    """
    def __init__(self, base_params):
        self.params = base_params
        self.macro_df = base_params['macro_df'].copy()

    def run_simulation(self, annual_operating_allowance=2400.0, gdp_shock_vector=None):
        """
        Runs a 10-year forward simulation of NZ Crown finances.

        Parameters:
        - annual_operating_allowance: New operating spending added each Budget ($ millions)
        - gdp_shock_vector: Optional array of additive shocks to real GDP growth rates
        """
        df = self.macro_df.copy()
        n = len(df)

        if gdp_shock_vector is not None:
            df['Real_GDP_Growth'] += gdp_shock_vector
            df['Nominal_GDP_Growth'] = (1 + df['Real_GDP_Growth']) * (1 + df['Inflation']) - 1

        # Initialize arrays
        gdp = np.zeros(n)
        tax_rev = np.zeros(n)
        nontax_rev = np.zeros(n)
        base_exp = np.zeros(n)
        cum_allowance = np.zeros(n)
        debt_servicing = np.zeros(n)
        total_exp = np.zeros(n)
        obegal = np.zeros(n)
        net_debt = np.zeros(n)
        net_debt_pct_gdp = np.zeros(n)

        # Base Year (t=0)
        gdp[0] = self.params['base_gdp']
        tax_rev[0] = self.params['base_tax_revenue']
        nontax_rev[0] = self.params['base_nontax_revenue']
        base_exp[0] = self.params['base_expenses']
        cum_allowance[0] = 0.0
        net_debt[0] = self.params['base_net_debt']
        debt_servicing[0] = net_debt[0] * df.loc[0, 'Effective_Interest_Rate']
        total_exp[0] = base_exp[0] + debt_servicing[0]
        obegal[0] = (tax_rev[0] + nontax_rev[0]) - total_exp[0]
        net_debt_pct_gdp[0] = (net_debt[0] / gdp[0]) * 100

        # Iterative Projection Engine (t=1 to 10)
        for t in range(1, n):
            nom_gdp_growth = df.loc[t, 'Nominal_GDP_Growth']
            inflation = df.loc[t, 'Inflation']
            i_rate = df.loc[t, 'Effective_Interest_Rate']

            # GDP Projection
            gdp[t] = gdp[t-1] * (1 + nom_gdp_growth)

            # Tax Revenue via Buoyancy Model
            tax_growth = nom_gdp_growth * self.params['tax_buoyancy']
            tax_rev[t] = tax_rev[t-1] * (1 + tax_growth)

            # Non-tax Revenue indexed to inflation
            nontax_rev[t] = nontax_rev[t-1] * (1 + inflation)

            # Base Expenses indexed to CPI inflation + demographic baseline (1%)
            base_exp[t] = base_exp[t-1] * (1 + inflation + 0.010)

            # Cumulative Operating Allowances (Added every year)
            cum_allowance[t] = cum_allowance[t-1] + annual_operating_allowance

            # Debt Servicing Costs on previous period debt
            debt_servicing[t] = net_debt[t-1] * i_rate

            # Total Core Crown Expenses
            total_exp[t] = base_exp[t] + cum_allowance[t] + debt_servicing[t]

            # OBEGAL Operating Balance
            obegal[t] = (tax_rev[t] + nontax_rev[t]) - total_exp[t]

            # Net Debt Accumulation (Deficit + Capital Allocations)
            net_debt[t] = net_debt[t-1] - obegal[t] + self.params['capital_allowance_annual']
            net_debt_pct_gdp[t] = (net_debt[t] / gdp[t]) * 100

        df['Nominal_GDP'] = gdp
        df['Tax_Revenue'] = tax_rev
        df['Total_Revenue'] = tax_rev + nontax_rev
        df['Base_Expenses'] = base_exp
        df['Cumulative_Allowance'] = cum_allowance
        df['Debt_Servicing'] = debt_servicing
        df['Total_Expenses'] = total_exp
        df['OBEGAL'] = obegal
        df['Net_Core_Crown_Debt'] = net_debt
        df['Net_Debt_Pct_GDP'] = net_debt_pct_gdp

        return df

print("NZFiscalStrategyModel Class initialized.")

NZFiscalStrategyModel Class initialized.


In [4]:
# Cell 5: Scenario Parameterization & Engine Execution

model = NZFiscalStrategyModel(baseline_data)

# Define Fiscal Policy Allowance Scenarios
scenarios = {
    'Fiscal Consolidation ($1.5B Allowance)': 1500.0,
    'Baseline Policy Target ($2.4B Allowance)': 2400.0,
    'Moderate Expansion ($3.5B Allowance)': 3500.0,
    'High Expenditure ($5.0B Allowance)': 5000.0
}

scenario_results = {}
for name, allowance in scenarios.items():
    scenario_results[name] = model.run_simulation(annual_operating_allowance=allowance)

# Perform Stochastic Monte Carlo Simulation (Macroeconomic GDP Shock Vulnerability)
np.random.seed(42)
n_simulations = 1000
mc_years = baseline_data['macro_df']['Fiscal_Year'].values
mc_debt_paths = np.zeros((n_simulations, len(mc_years)))
mc_obegal_paths = np.zeros((n_simulations, len(mc_years)))

for i in range(n_simulations):
    # Generate correlated real GDP shocks across 10 years (std dev = 1.5%)
    shocks = np.random.normal(loc=0.0, scale=0.015, size=len(mc_years))
    res = model.run_simulation(annual_operating_allowance=2400.0, gdp_shock_vector=shocks)
    mc_debt_paths[i, :] = res['Net_Debt_Pct_GDP'].values
    mc_obegal_paths[i, :] = res['OBEGAL'].values / 1000.0 # In $ Billions

print("Deterministic Policy Scenarios & Monte Carlo Simulations Executed Successfully.")

Deterministic Policy Scenarios & Monte Carlo Simulations Executed Successfully.


In [5]:
# Cell 6: Interactive Publication-Grade Visualizations (Plotly)

# Figure 1: OBEGAL Trajectories across Operating Allowance Scenarios
fig_obegal = go.Figure()

colors = {'Fiscal Consolidation ($1.5B Allowance)': '#008080',
          'Baseline Policy Target ($2.4B Allowance)': '#003366',
          'Moderate Expansion ($3.5B Allowance)': '#E65100',
          'High Expenditure ($5.0B Allowance)': '#B71C1C'}

for name, df in scenario_results.items():
    fig_obegal.add_trace(go.Scatter(
        x=df['Fiscal_Year'],
        y=df['OBEGAL'] / 1000.0,
        mode='lines+markers',
        name=name,
        line=dict(color=colors[name], width=3)
    ))

fig_obegal.add_hline(y=0, line_dash="dash", line_color="black", annotation_text="OBEGAL Surplus Threshold ($0B)")

fig_obegal.update_layout(
    title="<b>New Zealand Treasury FSM: OBEGAL Path under Operating Allowance Scenarios</b>",
    xaxis_title="Fiscal Year (Ending June)",
    yaxis_title="OBEGAL Balance ($ NZD Billions)",
    template="plotly_white",
    hovermode="x unified",
    legend=dict(yanchor="top", y=0.99, xanchor="left", x=0.01)
)

fig_obegal.show()

# Figure 2: Net Core Crown Debt (% of GDP) vs Target Ceiling
fig_debt = go.Figure()

for name, df in scenario_results.items():
    fig_debt.add_trace(go.Scatter(
        x=df['Fiscal_Year'],
        y=df['Net_Debt_Pct_GDP'],
        mode='lines+markers',
        name=name,
        line=dict(color=colors[name], width=3)
    ))

# Treasury Fiscal Strategy Anchor Target (e.g., 40% Debt Cap)
fig_debt.add_hline(y=40.0, line_dash="dot", line_color="red", annotation_text="Fiscal Anchor Limit (40% Net Debt-to-GDP)")

fig_debt.update_layout(
    title="<b>Net Core Crown Debt Trajectory (% of Nominal GDP)</b>",
    xaxis_title="Fiscal Year (Ending June)",
    yaxis_title="Net Core Crown Debt (% of GDP)",
    template="plotly_white",
    hovermode="x unified",
    legend=dict(yanchor="top", y=0.99, xanchor="left", x=0.01)
)

fig_debt.show()

# Figure 3: Monte Carlo Fan Chart for Net Core Crown Debt (% GDP)
p10 = np.percentile(mc_debt_paths, 10, axis=0)
p25 = np.percentile(mc_debt_paths, 25, axis=0)
p50 = np.percentile(mc_debt_paths, 50, axis=0)
p75 = np.percentile(mc_debt_paths, 75, axis=0)
p90 = np.percentile(mc_debt_paths, 90, axis=0)

fig_fan = go.Figure()

# 10th - 90th percentile band
fig_fan.add_trace(go.Scatter(
    x=np.concatenate([mc_years, mc_years[::-1]]),
    y=np.concatenate([p90, p10[::-1]]),
    fill='todense',
    fillcolor='rgba(0, 51, 102, 0.15)',
    line=dict(color='rgba(255,255,255,0)'),
    hoverinfo="skip",
    showlegend=True,
    name='10th-90th Percentile Risk Band'
))

# 25th - 75th percentile band
fig_fan.add_trace(go.Scatter(
    x=np.concatenate([mc_years, mc_years[::-1]]),
    y=np.concatenate([p75, p25[::-1]]),
    fill='todense',
    fillcolor='rgba(0, 51, 102, 0.30)',
    line=dict(color='rgba(255,255,255,0)'),
    hoverinfo="skip",
    showlegend=True,
    name='25th-75th Percentile Confidence Band'
))

# Median path
fig_fan.add_trace(go.Scatter(
    x=mc_years,
    y=p50,
    mode='lines+markers',
    line=dict(color='#003366', width=3.5),
    name='Median Stochastic Path (Baseline $2.4B Allowance)'
))

fig_fan.update_layout(
    title="<b>Stochastic Debt Sustainability Analysis (S-DSA) - 1,000 GDP Shock Simulations</b>",
    xaxis_title="Fiscal Year",
    yaxis_title="Net Core Crown Debt (% of GDP)",
    template="plotly_white",
    hovermode="x unified"
)

fig_fan.show()

ValueError: 
    Invalid value of type 'builtins.str' received for the 'fill' property of scatter
        Received value: 'todense'

    The 'fill' property is an enumeration that may be specified as:
      - One of the following enumeration values:
            ['none', 'tozeroy', 'tozerox', 'tonexty', 'tonextx',
            'toself', 'tonext']

In [ ]:
# Cell 7: Policy Matrix & Sensitivity Table Generation

summary_records = []

for name, df in scenario_results.items():
    # Identify fiscal year of return to surplus
    surplus_df = df[df['OBEGAL'] >= 0]
    surplus_year = surplus_df['Fiscal_Year'].iloc[0] if len(surplus_df) > 0 else "No Surplus by 2036"

    peak_debt_pct = df['Net_Debt_Pct_GDP'].max()
    final_debt_pct = df['Net_Debt_Pct_GDP'].iloc[-1]
    final_obegal_bn = df['OBEGAL'].iloc[-1] / 1000.0

    summary_records.append({
        'Policy Scenario': name,
        'Return to OBEGAL Surplus': surplus_year,
        'Peak Net Debt (% GDP)': f"{peak_debt_pct:.1f}%",
        '2036 Net Debt (% GDP)': f"{final_debt_pct:.1f}%",
        '2036 OBEGAL ($ NZD Bn)': f"${final_obegal_bn:+.2f}B"
    })

summary_table = pd.DataFrame(summary_records)

print("="*85)
print("NZ TREASURY FISCAL STRATEGY MODEL - EXECUTIVE POLICY MATRIX")
print("="*85)
print(summary_table.to_string(index=False))
print("="*85)

### Project Summary & Executive Takeaway

This project models sovereign macroeconomic decisions for public sector fiscal management. By evaluating new Budget operating allowances, tax revenue buoyancies, and stochastic GDP growth shocks across a 10-year horizon, the tool simulates the trajectory of Aotearoa New Zealand's OBEGAL balance and Net Core Crown Debt, automatically identifying fiscal pathways that return the Crown to surplus versus those that result in unsustainable debt trajectories.

#### Resume-Ready Portfolio Bullet
* **Developed** a Python-based quantitative simulation model to evaluate New Zealand’s Net Core Crown Debt trajectory and OBEGAL balance across multi-year forecast horizons under varying budget spending allowances and macroeconomic shock scenarios.